# AI & Machine Learning – Task 2
## Feature Engineering, Model Optimization & Performance Comparison
**Maincrafts Technology**

---

### Objective
Build an enhanced House Price Prediction system using:
- Feature-scaled input data
- Multiple regression models
- Structured model performance comparison
- A final selected model with proper justification


## Step 1: Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, r2_score

print("✅ All libraries imported successfully.")


## Step 2: Load the Dataset

In [ ]:
# Load California Housing Dataset
data = fetch_california_housing(as_frame=True)
df = pd.concat([data.data, data.target.rename("HousePrice")], axis=1)

print(f"Dataset shape: {df.shape}")
print(f"\nFeatures: {list(data.feature_names)}")
print(f"Target: HousePrice (Median House Value)")
df.head()


## Step 2b: Exploratory Data Analysis

In [ ]:
# Check for missing values and data types
print("Dataset Info:")
print(df.info())
print("\nMissing Values:")
print(df.isnull().sum())
print("\nBasic Statistics:")
df.describe()


In [ ]:
# Distribution of target variable
plt.figure(figsize=(8, 4))
plt.hist(df['HousePrice'], bins=50, color='steelblue', edgecolor='black', alpha=0.7)
plt.xlabel('Median House Price (in $100,000s)')
plt.ylabel('Frequency')
plt.title('Distribution of House Prices')
plt.tight_layout()
plt.show()


In [ ]:
# Correlation heatmap
plt.figure(figsize=(10, 6))
sns.heatmap(df.corr(), annot=True, fmt='.2f', cmap='coolwarm', square=True)
plt.title('Feature Correlation Heatmap')
plt.tight_layout()
plt.show()


## Step 3: Separate Features and Target Variable

In [ ]:
X = df.drop("HousePrice", axis=1)
y = df["HousePrice"]

print(f"Features shape: {X.shape}")
print(f"Target shape:   {y.shape}")


## Step 4: Feature Scaling (Critical Step)

Features exist on different numeric ranges. Without scaling:
- Some features dominate others
- Model performance becomes unstable

We use **StandardScaler** to normalize all features to a common scale.


In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Before Scaling (first row):")
print(X.iloc[0].values)
print("\nAfter Scaling (first row):")
print(X_scaled[0])
print("\n✅ Feature scaling applied successfully.")


## Step 5: Train–Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

print(f"Training samples: {X_train.shape[0]}")
print(f"Testing samples:  {X_test.shape[0]}")


## Step 6: Train Multiple Models

| Model | Rationale |
|-------|-----------|
| Linear Regression | Baseline model |
| Ridge Regression | Reduces overfitting via L2 regularization |
| Decision Tree | Captures non-linear relationships |


In [ ]:
models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(alpha=1.0),
    "Decision Tree": DecisionTreeRegressor(max_depth=5)
}

print("✅ Models defined:")
for name in models:
    print(f"  - {name}")


## Step 7: Model Evaluation and Comparison

**Metrics used:**
- **RMSE** – Root Mean Squared Error (lower is better)
- **R² Score** – Coefficient of Determination (higher is better, max = 1.0)


In [ ]:
results = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    
    rmse = mean_squared_error(y_test, predictions, squared=False)
    r2   = r2_score(y_test, predictions)
    
    results[name] = {"RMSE": round(rmse, 4), "R2 Score": round(r2, 4)}
    print(f"{name:25s} | RMSE: {rmse:.4f} | R²: {r2:.4f}")

results_df = pd.DataFrame(results).T
print("\n--- Model Performance Summary ---")
results_df


In [ ]:
# Bar chart comparison
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

results_df['RMSE'].plot(kind='bar', ax=axes[0], color='tomato', edgecolor='black')
axes[0].set_title('RMSE Comparison (Lower is Better)')
axes[0].set_ylabel('RMSE')
axes[0].set_xticklabels(results_df.index, rotation=15)

results_df['R2 Score'].plot(kind='bar', ax=axes[1], color='steelblue', edgecolor='black')
axes[1].set_title('R² Score Comparison (Higher is Better)')
axes[1].set_ylabel('R² Score')
axes[1].set_xticklabels(results_df.index, rotation=15)

plt.tight_layout()
plt.show()


## Step 8: Visual Performance Validation

In [ ]:
# Actual vs Predicted for all models
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, (name, model) in zip(axes, models.items()):
    y_pred = model.predict(X_test)
    ax.scatter(y_test, y_pred, alpha=0.3, color='steelblue', s=10)
    ax.plot([y_test.min(), y_test.max()],
            [y_test.min(), y_test.max()], 'r--', linewidth=2)
    ax.set_xlabel("Actual House Price")
    ax.set_ylabel("Predicted House Price")
    ax.set_title(f"{name}\nR²={r2_score(y_test, y_pred):.3f}")

plt.suptitle("Actual vs Predicted House Prices – All Models", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


## Step 9: Best Model Selection & Conclusion

In [ ]:
best_model_name = results_df['R2 Score'].idxmax()
best_r2   = results_df.loc[best_model_name, 'R2 Score']
best_rmse = results_df.loc[best_model_name, 'RMSE']

print("=" * 50)
print(f"  Best Model  : {best_model_name}")
print(f"  R² Score    : {best_r2}")
print(f"  RMSE        : {best_rmse}")
print("=" * 50)
print(f"\n✅ '{best_model_name}' selected as the best-performing model.")
print("   Justification: Highest R² and competitive RMSE on unseen test data.")


## (Optional) Step 10: Save the Best Model

In [ ]:
import joblib

best_model_obj = models[best_model_name]
joblib.dump(best_model_obj, "best_model.pkl")
joblib.dump(scaler, "scaler.pkl")

print(f"✅ Best model saved as 'best_model.pkl'")
print(f"✅ Scaler saved as 'scaler.pkl'")
